# LeetCode 2067 — Number of Equal Count Substrings bằng Python

Link đề: https://leetcode.com/problems/number-of-equal-count-substrings/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window, Counting

> **Lưu ý: đây là bài Premium** trên LeetCode — cần tài khoản Premium để xem đề gốc và nộp bài. Đề dưới đây viết theo hiểu biết về bài (nguồn: bản đề của doocs/leetcode); nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc**.

Skeleton LeetCode:

```python
class Solution:
    def equalCountSubstrings(self, s: str, count: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Khai thác cấu trúc: chuỗi con "đếm đều" có `u` chữ cái khác nhau thì **độ dài đúng bằng `u · count`** ⇒ chỉ có **26** độ dài khả dĩ.
2. Với mỗi độ dài cố định, dùng **cửa sổ trượt kích thước cố định** và duy trì "số chữ cái có tần suất đúng bằng `count`" trong `O(1)` mỗi bước.
3. Điều kiện cửa sổ hợp lệ: số chữ cái có tần suất `= count` bằng **đúng** `u` (khi đó mọi chữ cái có mặt đều có tần suất `count`).
4. Tổng độ phức tạp `O(26 · n)`; so với vét cạn `O(n² · 26)`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` (chữ thường) và số nguyên `count`. Chuỗi con là "**đếm đều**" (equal count) nếu **mỗi** chữ cái **xuất hiện** trong nó xuất hiện **đúng** `count` lần. Trả về số chuỗi con đếm đều của `s` (chuỗi con phân biệt theo vị trí).

**Ví dụ 1:** `s = "aaabcbbcc", count = 3` → `3` (`"aaa"`, `"bcbbcc"`, `"aaabcbbcc"`).

**Ví dụ 2:** `s = "abcd", count = 2` → `0`.

**Ví dụ 3:** `s = "a", count = 5` → `0`.

**Ràng buộc:** `1 ≤ len(s) ≤ 3·10⁴`; `1 ≤ count ≤ 3·10⁴`; chữ thường.

*(Đề Premium — đã diễn đạt lại, hãy đối chiếu lại đề gốc.)*

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [(("aaabcbbcc", 3), 3), (("abcd", 2), 0), (("a", 5), 0)]
for (s, c), exp in examples:
    print(f"s={s!r}, count={c} -> kỳ vọng {exp}")

s='aaabcbbcc', count=3 -> kỳ vọng 3
s='abcd', count=2 -> kỳ vọng 0
s='a', count=5 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Độ dài của một chuỗi con đếm đều

Nếu chuỗi con có `u` chữ cái khác nhau, mỗi chữ xuất hiện `count` lần ⇒ độ dài `= u · count`. Vì `1 ≤ u ≤ 26`, chỉ có tối đa 26 độ dài cần xét (và chỉ những độ dài `≤ n`).

In [2]:
s, c = "aaabcbbcc", 3
print("các độ dài khả dĩ:", [u * c for u in range(1, 27) if u * c <= len(s)])

các độ dài khả dĩ: [3, 6, 9]


### 3.2 Cửa sổ trượt kích thước cố định

Duy trì tần suất `cnt[ch]` trong cửa sổ `[i − L + 1, i]`; khi trượt, thêm ký tự mới và bớt ký tự rơi ra — mỗi bước `O(1)`.

### 3.3 Đếm "số chữ cái đạt đúng ngưỡng"

Biến `exact` = số chữ cái có `cnt = count`. Khi `cnt[ch]` đổi từ `a` sang `a ± 1`, chỉ cần kiểm tra `a == count` (mất) và `a ± 1 == count` (thêm) để cập nhật `exact` trong `O(1)`.

In [3]:
cnt, exact, K = Counter(), 0, 2
for ch in "abab":
    if cnt[ch] == K: exact -= 1
    cnt[ch] += 1
    if cnt[ch] == K: exact += 1
    print(f"thêm {ch}: cnt={dict(cnt)}, exact={exact}")

thêm a: cnt={'a': 1}, exact=0
thêm b: cnt={'a': 1, 'b': 1}, exact=0
thêm a: cnt={'a': 2, 'b': 1}, exact=1
thêm b: cnt={'a': 2, 'b': 2}, exact=2


### 3.4 Điều kiện hợp lệ của cửa sổ độ dài `u · count`

Nếu `exact = u` thì `u` chữ cái chiếm `u · count` vị trí = toàn bộ cửa sổ ⇒ **không còn** chữ cái nào khác ⇒ cửa sổ đếm đều. Ngược lại nếu cửa sổ đếm đều thì có đúng `u` chữ cái, mỗi chữ `count` lần ⇒ `exact = u`.

## 4. Ý tưởng nền tảng / trực giác

Thay vì xét `n(n+1)/2` chuỗi con, xét theo **số chữ cái khác nhau** `u`: chuỗi con hợp lệ phải có độ dài `L = u·count`. Với mỗi `u`, trượt cửa sổ độ dài `L` và đếm các vị trí mà `exact == u`.

Tính tay **Ví dụ 1** (`count = 3`, `n = 9`):

| u | L | cửa sổ hợp lệ |
|---|---|---|
| 1 | 3 | `aaa` (vị trí 0..2) |
| 2 | 6 | `bcbbcc` (3..8) |
| 3 | 9 | `aaabcbbcc` (0..8) |

Tổng `3` ✓.

In [4]:
s, c = "aaabcbbcc", 3; n = len(s)
for u in range(1, 4):
    L = u * c
    ok = [s[i:i + L] for i in range(n - L + 1) if all(v == c for v in Counter(s[i:i + L]).values())]
    print(f"u={u}, L={L}: {ok}")

u=1, L=3: ['aaa']
u=2, L=6: ['bcbbcc']
u=3, L=9: ['aaabcbbcc']


## 5. Cách 1 — Brute Force / baseline: mọi chuỗi con, cập nhật tần suất dần

In [5]:
def equal_bruteforce(s, count):
    n = len(s); total = 0
    for l in range(n):
        cnt = [0] * 26
        for r in range(l, n):
            cnt[ord(s[r]) - 97] += 1
            if all(x == 0 or x == count for x in cnt):
                total += 1
    return total

for (s_, c), exp in examples:
    got = equal_bruteforce(s_, c); print(got, "kỳ vọng", exp); assert got == exp

3 kỳ vọng 3
0 kỳ vọng 0
0 kỳ vọng 0


**Độ phức tạp:** `O(n² · 26)` = `2.3·10¹⁰` với `n = 3·10⁴` ⇒ TLE.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** độ dài hợp lệ chỉ thuộc `{count, 2·count, …, 26·count}`.

**Insight 2:** với độ dài cố định `L = u·count`, cửa sổ hợp lệ ⇔ `exact == u`.

**Insight 3:** cập nhật `exact` `O(1)` mỗi lần thêm/bớt ký tự.

Tổng: `Σ_u O(n) = O(26n)`.

**Invariant (mỗi `u`):** sau khi xử lý ký tự `i`, `cnt` là tần suất của `s[max(0, i−L+1) .. i]` và `exact` là số chữ có tần suất đúng `count` trong đó.

### 6.1 Minh họa: `exact` trong lúc trượt cửa sổ `L = 6` trên Ví dụ 1

In [6]:
s, c, u = "aaabcbbcc", 3, 2; L = u * c
cnt = Counter(); exact = 0
for i, ch in enumerate(s):
    if cnt[ch] == c: exact -= 1
    cnt[ch] += 1
    if cnt[ch] == c: exact += 1
    if i >= L:
        out = s[i - L]
        if cnt[out] == c: exact -= 1
        cnt[out] -= 1
        if cnt[out] == c: exact += 1
    if i >= L - 1:
        print(f"cửa sổ {s[i - L + 1:i + 1]}: exact = {exact} {'-> hợp lệ' if exact == u else ''}")

cửa sổ aaabcb: exact = 1 
cửa sổ aabcbb: exact = 1 
cửa sổ abcbbc: exact = 1 
cửa sổ bcbbcc: exact = 2 -> hợp lệ


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt cho mỗi `u` (bản nộp)

In [7]:
class Solution:
    def equalCountSubstrings(self, s: str, count: int) -> int:
        n = len(s)
        a = [ord(ch) - 97 for ch in s]
        ans = 0
        for u in range(1, 27):
            L = u * count
            if L > n:
                break
            cnt = [0] * 26
            exact = 0
            for i in range(n):
                x = a[i]
                if cnt[x] == count: exact -= 1
                cnt[x] += 1
                if cnt[x] == count: exact += 1
                if i >= L:
                    y = a[i - L]
                    if cnt[y] == count: exact -= 1
                    cnt[y] -= 1
                    if cnt[y] == count: exact += 1
                if i >= L - 1 and exact == u:
                    ans += 1
        return ans

print([Solution().equalCountSubstrings(*a) for a, _ in examples])

[3, 0, 0]


**Độ phức tạp Cách 2:** `O(26 · n)` thời gian, `O(26)` bộ nhớ.

### 7.2 Cách 3 — Chỉ duyệt các `u` không vượt số chữ cái khác nhau của `s`

Nếu `s` chỉ có `D` chữ cái khác nhau thì `u ≤ D`. Thêm điều kiện này giúp nhanh hơn khi bảng chữ cái nhỏ.

In [8]:
def equal_distinct_bound(s, count):
    D = len(set(s)); n = len(s); total = 0
    for u in range(1, D + 1):
        L = u * count
        if L > n: break
        cnt = Counter(s[:L]); exact = sum(1 for v in cnt.values() if v == count)
        total += exact == u
        for i in range(L, n):
            for ch, d in ((s[i], 1), (s[i - L], -1)):
                if cnt[ch] == count: exact -= 1
                cnt[ch] += d
                if cnt[ch] == count: exact += 1
            total += exact == u
    return total

print([equal_distinct_bound(*a) for a, _ in examples])

[3, 0, 0]


### 7.3 Cách 4 — Tiền tố tần suất + kiểm tra mỗi cửa sổ `O(26)`

Tính `P[i][c]` = số lần chữ `c` trong `s[:i]`. Với mỗi `u` và mỗi cửa sổ, kiểm tra 26 tần suất. `O(26 · 26 · n)` — chậm hơn Cách 2 nhưng đơn giản, ít lỗi cập nhật.

In [9]:
def equal_prefix(s, count):
    n = len(s); P = [[0] * 26]
    for ch in s:
        row = P[-1][:]; row[ord(ch) - 97] += 1; P.append(row)
    total = 0
    for u in range(1, 27):
        L = u * count
        if L > n: break
        for i in range(n - L + 1):
            if all(P[i + L][c] - P[i][c] in (0, count) for c in range(26)):
                total += 1
    return total

print([equal_prefix(*a) for a, _ in examples])

[3, 0, 0]


## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** — in các cửa sổ hợp lệ theo `u`:

In [10]:
def trace(s, count):
    n = len(s); ans = 0
    for u in range(1, 27):
        L = u * count
        if L > n: print(f"u={u}: L={L} > n, dừng"); break
        cnt = Counter(); exact = 0; found = []
        for i, ch in enumerate(s):
            if cnt[ch] == count: exact -= 1
            cnt[ch] += 1
            if cnt[ch] == count: exact += 1
            if i >= L:
                o = s[i - L]
                if cnt[o] == count: exact -= 1
                cnt[o] -= 1
                if cnt[o] == count: exact += 1
            if i >= L - 1 and exact == u: found.append((i - L + 1, i))
        ans += len(found); print(f"u={u}, L={L}: cửa sổ hợp lệ {found}")
    return ans

assert trace("aaabcbbcc", 3) == 3

u=1, L=3: cửa sổ hợp lệ [(0, 2)]
u=2, L=6: cửa sổ hợp lệ [(3, 8)]
u=3, L=9: cửa sổ hợp lệ [(0, 8)]
u=4: L=12 > n, dừng


| u | L | cửa sổ (đầu, cuối) |
|---|---|---|
| 1 | 3 | (0, 2) |
| 2 | 6 | (3, 8) |
| 3 | 9 | (0, 8) |
| 4 | 12 > 9 | dừng |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2**.

In [11]:
class Solution:
    def equalCountSubstrings(self, s: str, count: int) -> int:
        n = len(s)
        a = [ord(ch) - 97 for ch in s]
        ans = 0
        for u in range(1, 27):
            L = u * count
            if L > n:
                break
            cnt = [0] * 26
            exact = 0
            for i in range(n):
                x = a[i]
                if cnt[x] == count: exact -= 1
                cnt[x] += 1
                if cnt[x] == count: exact += 1
                if i >= L:
                    y = a[i - L]
                    if cnt[y] == count: exact -= 1
                    cnt[y] -= 1
                    if cnt[y] == count: exact += 1
                if i >= L - 1 and exact == u:
                    ans += 1
        return ans

for (s_, c), exp in examples:
    got = Solution().equalCountSubstrings(s_, c); print(got, exp); assert got == exp

3 3
0 0
0 0


## 10. Vì sao thuật toán đúng

**Phân hoạch theo `u`.** Mỗi chuỗi con đếm đều có đúng một giá trị `u` (số chữ cái khác nhau) và độ dài `u·count`. Vì vậy tập chuỗi con đếm đều được phân hoạch theo `u ∈ [1, 26]`, và với mỗi `u` chỉ cần xét các cửa sổ độ dài `u·count`.

**Kiểm tra cửa sổ.** Với cửa sổ độ dài `L = u·count`: nếu `exact = u`, các chữ cái đạt `count` chiếm `u·count = L` vị trí ⇒ không còn chữ nào khác ⇒ đếm đều. Ngược lại, cửa sổ đếm đều có `u'` chữ cái với `u'·count = L` ⇒ `u' = u` và tất cả đạt `count` ⇒ `exact = u`.

**Duy trì `exact`.** Khi `cnt[x]` thay đổi 1 đơn vị, trạng thái "bằng `count`" của **chỉ** chữ `x` có thể đổi; ta trừ trước khi đổi (nếu đang bằng) và cộng sau khi đổi (nếu thành bằng) ⇒ `exact` luôn đúng. ∎

### 10.1 Kiểm chứng thực nghiệm

In [12]:
def by_definition(s, count):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n) if all(v == count for v in Counter(s[l:r + 1]).values()))

rng = random.Random(1010)
for _ in range(500):
    s_ = "".join(rng.choice(rng.choice(["ab", "abc", "aab"])) for _ in range(rng.randint(1, 18))); c = rng.randint(1, 4)
    ref = by_definition(s_, c)
    assert ref == equal_bruteforce(s_, c) == Solution().equalCountSubstrings(s_, c) == equal_distinct_bound(s_, c) == equal_prefix(s_, c)
print("Định nghĩa = 4 cách trên 500 chuỗi")

Định nghĩa = 4 cách trên 500 chuỗi


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi chuỗi con | `O(26 n²)` | `O(26)` |
| 2. Cửa sổ mỗi `u` | `O(26 n)` | `O(26)` |
| 3. Như 2, `u ≤ D` | `O(D n)` | `O(D)` |
| 4. Tiền tố + kiểm 26 | `O(26² n)` | `O(26 n)` |

In [13]:
rng = random.Random(2067)
big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(3 * 10 ** 4))
for name, f in [("submit", Solution().equalCountSubstrings), ("distinct", equal_distinct_bound)]:
    t = time.perf_counter(); r = f(big, 1); print(f"{name:<9} count=1: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")
t = time.perf_counter(); equal_bruteforce(big[:800], 1); print(f"brute n = 800: {(time.perf_counter() - t) * 1000:.0f} ms")

submit    count=1: 180708 (133 ms)


distinct  count=1: 180708 (271 ms)
brute n = 800: 60 ms


## 12. Bộ test

In [14]:
ALL = {"submit": Solution().equalCountSubstrings, "distinct": equal_distinct_bound, "prefix": equal_prefix}
def run_all(s_, c, brute=True):
    res = {k: f(s_, c) for k, f in ALL.items()}
    if brute:
        res["brute"] = equal_bruteforce(s_, c)
    assert len(set(res.values())) == 1, (s_, c, res)
    return res["submit"]

for (s_, c), exp in examples:
    assert run_all(s_, c) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [(("a", 1), 1), (("aa", 1), 2), (("ab", 1), 3), (("abab", 2), 1), (("aabb", 2), 3), (("zzzz", 2), 3), (("abc", 1), 6)]
for (s_, c), exp in edge:
    assert run_all(s_, c) == exp, (s_, c, run_all(s_, c))
rng = random.Random(20670)
for _ in range(300):
    run_all("".join(rng.choice("abcd") for _ in range(rng.randint(1, 40))), rng.randint(1, 5))
print("Ví dụ, 7 edge case, 300 chuỗi ngẫu nhiên: OK")

Ví dụ, 7 edge case, 300 chuỗi ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- `count = 1`: đáp án là số chuỗi con có **mọi ký tự khác nhau**.
- `count > n` ⇒ đáp án 0.
- Đảo ngược chuỗi không đổi đáp án.

In [15]:
rng = random.Random(121)
f = Solution().equalCountSubstrings
for _ in range(500):
    s_ = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 25))); n = len(s_)
    assert f(s_, 1) == sum(1 for l in range(n) for r in range(l, n) if len(set(s_[l:r + 1])) == r - l + 1)
    assert f(s_, n + 1) == 0 and f(s_[::-1], 2) == f(s_, 2)
print("3 tính chất đúng trên 500 chuỗi")

3 tính chất đúng trên 500 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra cả các chữ cái **không xuất hiện** (đòi mọi chữ cái đều `count`)

In [16]:
def wrong_all_letters(s, count):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n) if all(Counter(s[l:r + 1])[ch] == count for ch in set(s)))   # SAI

for (s_, c), exp in examples:
    print(f"{s_}: sai {wrong_all_letters(s_, c)} | đúng {exp}")

aaabcbbcc: sai 1 | đúng 3
abcd: sai 0 | đúng 0
a: sai 0 | đúng 0


**Vì sao sai:** điều kiện chỉ áp dụng cho chữ cái **có mặt trong chuỗi con**; `"aaa"` hợp lệ dù không có `b`, `c`. **Sửa:** chấp nhận tần suất `0` hoặc `count`.

### Lỗi 2 — Dùng `≥ count` thay vì `== count`

In [17]:
def wrong_ge(s, count):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n) if all(v >= count for v in Counter(s[l:r + 1]).values()))   # SAI

for (s_, c), exp in examples + [(("aaaa", 2), Solution().equalCountSubstrings("aaaa", 2))]:   # "aaaa" tự dựng (đối chiếu lại đề gốc)
    print(f"{s_}: sai {wrong_ge(s_, c)} | đúng {exp}")

aaabcbbcc: sai 3 | đúng 3
abcd: sai 0 | đúng 0
a: sai 0 | đúng 0
aaaa: sai 6 | đúng 3


**Vì sao sai:** "exactly count times". **Sửa:** `==`.

### Lỗi 3 — Cửa sổ biến thiên kiểu hai con trỏ "co khi vượt `count`"

In [18]:
def wrong_two_pointers(s, count):
    cnt = Counter(); l = 0; total = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while cnt[ch] > count:
            cnt[s[l]] -= 1; l += 1
        if all(v in (0, count) for v in cnt.values()):
            total += 1                                              # SAI: chỉ đếm một cửa sổ cho mỗi r
    return total

for (s_, c), exp in examples + [(("aabb", 2), 3)]:
    print(f"{s_}: sai {wrong_two_pointers(s_, c)} | đúng {exp if isinstance(exp, int) else exp}")

aaabcbbcc: sai 2 | đúng 3
abcd: sai 0 | đúng 0
a: sai 0 | đúng 0
aabb: sai 2 | đúng 3


**Vì sao sai:** với mỗi đầu phải `r` có thể có **nhiều** đầu trái hợp lệ (ví dụ `"aabb"`, `count = 2`: cả `"bb"` và `"aabb"` kết thúc ở cuối). Tính chất "hợp lệ" không đơn điệu theo độ dài cửa sổ. **Sửa:** cố định độ dài theo `u`.

### Lỗi 4 — Không dừng khi `u · count > n` (hoặc lặp `u` tới `n`)

In [19]:
def slow_all_lengths(s, count):
    n = len(s); total = 0
    for L in range(1, n + 1):                                      # thử MỌI độ dài
        for i in range(n - L + 1):
            if all(v == count for v in Counter(s[i:i + L]).values()): total += 1
    return total

s_ = "".join(random.Random(4).choice("ab") for _ in range(300))
for name, fn in [("mọi độ dài", slow_all_lengths), ("chỉ u*count", Solution().equalCountSubstrings)]:
    t = time.perf_counter(); r = fn(s_, 3); print(f"{name:<12}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")

mọi độ dài  : 298 (96 ms)
chỉ u*count : 298 (1 ms)


**Vì sao không tốt:** đúng nhưng `O(n³)`; chỉ những độ dài là bội của `count` (tối đa 26 giá trị) mới có thể hợp lệ. **Sửa:** duyệt `u`.

### Lỗi 5 — Cập nhật `exact` sai thứ tự (cộng/trừ sau khi đổi `cnt`)

In [20]:
def wrong_exact_order(s, count):
    n = len(s); ans = 0
    for u in range(1, 27):
        L = u * count
        if L > n: break
        cnt = Counter(); exact = 0
        for i, ch in enumerate(s):
            cnt[ch] += 1
            if cnt[ch] == count: exact += 1
            if cnt[ch] == count + 1: pass                           # SAI: quên trừ khi vượt count
            if i >= L:
                o = s[i - L]; cnt[o] -= 1
                if cnt[o] == count: exact += 1
                if cnt[o] == count - 1: exact -= 1
            if i >= L - 1 and exact == u: ans += 1
    return ans

for (s_, c), exp in examples + [(("aaaa", 2), 3)]:
    print(f"{s_}: sai {wrong_exact_order(s_, c)} | đúng {Solution().equalCountSubstrings(s_, c)}")

aaabcbbcc: sai 3 | đúng 3
abcd: sai 0 | đúng 0
a: sai 0 | đúng 0
aaaa: sai 1 | đúng 3


**Vì sao sai:** khi `cnt` vượt `count` (từ `count` lên `count + 1`), chữ đó **không còn** đạt ngưỡng ⇒ phải giảm `exact`. Quên bước này làm `exact` lớn hơn thực tế. **Sửa:** mẫu "trừ nếu đang bằng → đổi → cộng nếu thành bằng".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chuỗi con với ràng buộc "mọi chữ cái xuất hiện đúng `k` lần" ⇒ **cố định số chữ cái khác nhau** (tối đa 26) để có cửa sổ cố định.
- Biến đếm "số phần tử đạt ngưỡng" cập nhật `O(1)` (trừ trước, cộng sau).
- Ràng buộc không đơn điệu theo độ dài ⇒ hai con trỏ thường sai; cố định một tham số để có cửa sổ cố định.
- Liên quan: **395 Longest Substring with At Least K Repeating Characters** (cũng duyệt số chữ cái khác nhau).

## 15. Biến thể của bài

1. **395**: dài nhất với mọi chữ cái xuất hiện **ít nhất** `k` lần — duyệt `u` + cửa sổ biến thiên.
2. **1297 Maximum Number of Occurrences of a Substring**: cửa sổ cố định + đếm chuỗi con.
3. **Đếm chuỗi con mọi chữ cái xuất hiện cùng số lần** (không cố định `count`) — duyệt `u` và `count`, hoặc băm vector chênh lệch.
4. **Chỉ đếm chuỗi con có đúng `u₀` chữ cái** — một vòng cửa sổ.
5. **1371** (nguyên âm chẵn lần): mask tiền tố.

### 15.1 Code biến thể: 395 — dài nhất với mọi chữ cái `≥ k` lần (duyệt `u`)

In [21]:
def longest_substring_k(s, k):
    best = 0
    for u in range(1, len(set(s)) + 1):
        cnt = Counter(); l = 0; distinct = atleast = 0
        for r, ch in enumerate(s):
            if cnt[ch] == 0: distinct += 1
            cnt[ch] += 1
            if cnt[ch] == k: atleast += 1
            while distinct > u:
                o = s[l]
                if cnt[o] == k: atleast -= 1
                cnt[o] -= 1
                if cnt[o] == 0: distinct -= 1
                l += 1
            if distinct == u == atleast: best = max(best, r - l + 1)
    return best

def lsk_brute(s, k):
    n = len(s); return max([r - l + 1 for l in range(n) for r in range(l, n) if all(v >= k for v in Counter(s[l:r + 1]).values())], default=0)

rng = random.Random(395)
for _ in range(300):
    s_ = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15))); k = rng.randint(1, 3); assert longest_substring_k(s_, k) == lsk_brute(s_, k)
print("395 khớp vét cạn;", longest_substring_k("ababbc", 2), "(LeetCode: 5)")

395 khớp vét cạn; 5 (LeetCode: 5)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con | Vét cạn | `O(26n²)` | `O(26)` | Kiểm chứng |
| 2. Cửa sổ mỗi `u` | `exact == u` | `O(26n)` | `O(26)` | **Bản nộp** |
| 3. `u ≤ D` | Bớt vòng | `O(Dn)` | `O(D)` | Bảng chữ nhỏ |
| 4. Tiền tố | Kiểm 26 | `O(26²n)` | `O(26n)` | Ít lỗi cập nhật |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("ab", 1)`; (b) `("abab", 2)`; (c) `("aabb", 2)`; (d) `("zzzz", 2)`.

**Bài 2 (tính tay):** Với `s = "aabbab", count = 2`: liệt kê độ dài khả dĩ và các chuỗi con hợp lệ.

**Bài 3 (điền TODO):** Hoàn thiện `update(cnt, exact, x, delta, count)` trả `exact` mới.

**Bài 4 (tự viết lại):** Viết `equal_count_with_u(s, count, u)` chỉ đếm các chuỗi con có **đúng** `u` chữ cái khác nhau.

**Bài 5 (mini challenge):** Viết `all_equal_freq_substrings(s)`: đếm chuỗi con mà **mọi chữ cái có mặt có cùng tần suất** (tần suất nào cũng được). Gợi ý: duyệt `u` và `count` với `u·count ≤ n` (dùng Cách 2 cho mỗi cặp) — đối chiếu vét cạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def update(cnt, exact, x, delta, count):
    # TODO: nếu cnt[x] == count: exact -= 1; cnt[x] += delta; nếu cnt[x] == count: exact += 1; trả exact
    pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def equal_count_with_u(s, count, u):
    # TODO: một vòng cửa sổ độ dài u * count
    pass

def all_equal_freq_substrings(s):
    # TODO: cộng equal_count_with_u(s, c, u) với mọi u, c thỏa u * c <= len(s)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: độ dài là bội của `count`. Mức 2: kiểm từng cửa sổ. Mức 3: (a) `3` (`a`, `b`, `ab`); (b) `1` (`abab`); (c) `3` (`aa`, `bb`, `aabb`); (d) `3` (ba cửa sổ `zz`).

**Bài 2.** Mức 1: độ dài 2, 4, 6. Mức 2: độ dài 2: `aa`, `bb`; độ dài 4: `aabb` ✓, `abba` ✓, `bbab` ✗. Mức 3: độ dài 6: `aabbab`: `a` 3 lần ✗ ⇒ tổng `4`.

**Bài 3.** Mức 1: trừ trước khi đổi. Mức 2: đổi. Mức 3: cộng sau khi đổi.

**Bài 4.** Mức 1: `L = u·count`. Mức 2: cửa sổ trượt. Mức 3: đếm khi `exact == u`.

**Bài 5.** Mức 1: mỗi chuỗi con hợp lệ có duy nhất `(u, count)`. Mức 2: tổng qua cặp. Mức 3: `Σ_u (n/u)` cặp ⇒ `O(26 · n · log)` tổng.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
f = Solution().equalCountSubstrings
b1 = [f("ab", 1), f("abab", 2), f("aabb", 2), f("zzzz", 2)]
print("Bài 1:", b1); assert b1 == [3, 1, 3, 3]
print("Bài 2:", f("aabbab", 2), [s2 for L in (2, 4, 6) for s2 in ["aabbab"[i:i + L] for i in range(7 - L)] if all(v == 2 for v in Counter(s2).values())])
assert f("aabbab", 2) == 4
def update(cnt, exact, x, delta, count):
    if cnt[x] == count: exact -= 1
    cnt[x] += delta
    if cnt[x] == count: exact += 1
    return exact
cnt = Counter(); e = 0
for ch in "aab": e = update(cnt, e, ch, 1, 2)
assert e == 1; e = update(cnt, e, "a", 1, 2); assert e == 0
print("Bài 3: OK")

Bài 1: [3, 1, 3, 3]
Bài 2: 4 ['aa', 'bb', 'aabb', 'abba']
Bài 3: OK


In [25]:
# Bài 4, 5
def equal_count_with_u(s, count, u):
    n = len(s); L = u * count
    if L > n: return 0
    cnt = Counter(); exact = 0; total = 0
    for i, ch in enumerate(s):
        exact = update(cnt, exact, ch, 1, count)
        if i >= L: exact = update(cnt, exact, s[i - L], -1, count)
        if i >= L - 1 and exact == u: total += 1
    return total

def all_equal_freq_substrings(s):
    n = len(s)
    return sum(equal_count_with_u(s, c, u) for u in range(1, 27) for c in range(1, n // u + 1))

rng = random.Random(5)
for _ in range(200):
    s_ = "".join(rng.choice("abc") for _ in range(rng.randint(1, 14))); n = len(s_)
    for c in range(1, 4):
        assert sum(equal_count_with_u(s_, c, u) for u in range(1, 27)) == f(s_, c)
    assert all_equal_freq_substrings(s_) == sum(1 for l in range(n) for r in range(l, n) if len(set(Counter(s_[l:r + 1]).values())) == 1)
print("Bài 4, 5: OK |", equal_count_with_u("aaabcbbcc", 3, 2), all_equal_freq_substrings("aabb"))

Bài 4, 5: OK | 1 8


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
for u in range(1, 27):
    L = u * count
    if L > n: break
    cnt = [0] * 26; exact = 0
    for i in range(n):
        add a[i]  (trừ exact nếu đang == count, tăng, cộng nếu thành == count)
        if i >= L: remove a[i - L]  (tương tự)
        if i >= L - 1 and exact == u: ans += 1
```

**Checklist:**
- [ ] Chỉ xét độ dài `u·count`, `u = 1..26`, dừng khi `> n`?
- [ ] Hợp lệ ⇔ `exact == u`?
- [ ] Cập nhật `exact` đúng thứ tự (trừ trước, cộng sau)?
- [ ] Chữ cái vắng mặt không bị bắt phải đạt `count`?
- [ ] Không dùng hai con trỏ biến thiên?